Download EEG Spectrogram Dataset from Kaggle

In [4]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ayushaote/eeg-spectrogram-images-for-schizophrenia-detection")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'eeg-spectrogram-images-for-schizophrenia-detection' dataset.
Path to dataset files: /kaggle/input/eeg-spectrogram-images-for-schizophrenia-detection


Create Splits for Training, Test, and Val

In [5]:
import os
import shutil
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator

data_dir = "/kaggle/input/eeg-spectrogram-images-for-schizophrenia-detection/EEG_IMAGES_VGG16-20250724T160523Z-1-001/EEG_IMAGES_VGG16"
labels_df = pd.read_csv("/kaggle/input/eeg-spectrogram-images-for-schizophrenia-detection/labels.csv")
subject_df = labels_df.groupby("subject_id").first().reset_index()
split_dir = "SPLIT_IMAGES"

# Stratified splitting
train_subjects, test_subjects = train_test_split(
    subject_df,
    test_size=0.15,
    stratify=subject_df["label"],
    random_state=42
)
val_subjects, test_subjects = train_test_split(
    test_subjects,
    test_size=0.15,
    stratify=test_subjects["label"],
    random_state=42
)

# Create a subject_id → split mapping
split_map = {}
for sid in train_subjects["subject_id"]:
    split_map[sid] = "train"
for sid in val_subjects["subject_id"]:
    split_map[sid] = "val"
for sid in test_subjects["subject_id"]:
    split_map[sid] = "test"

# Create folders
for split in ["train", "val", "test"]:
    for label in ["Control", "Patient"]:
        os.makedirs(os.path.join(split_dir, split, label), exist_ok=True)

copied = 0
for _, row in labels_df.iterrows():
    sid = row["subject_id"]
    label = row["label"]
    split = split_map.get(sid)

    #while labels_df lists .edf our images are .png
    original_filename = row["filename"]
    base_filename, _ = os.path.splitext(original_filename)
    image_filename = base_filename + ".png"

    # Construct the full source path to the image file
    src = os.path.join(data_dir, image_filename)
    dst = os.path.join(split_dir, split, label)

    if os.path.exists(src):
        shutil.copy2(src, dst)
        copied += 1
    else:
        print(f"File not found: {src}")
        pass

print(f"Copied {copied} images into split folders at: {split_dir}")

# Set Parameters
img_size = (224, 224)
batch_size = 32

# Data generators
train_gen = ImageDataGenerator(rescale=1./255, zoom_range=0.2, horizontal_flip=True, rotation_range=15).flow_from_directory(
    os.path.join(split_dir, "train"),
    target_size=img_size,
    batch_size=batch_size,
    class_mode="categorical"
)

val_gen = ImageDataGenerator(rescale=1./255).flow_from_directory(
    os.path.join(split_dir, "val"),
    target_size=img_size,
    batch_size=batch_size,
    class_mode="categorical"
)

test_gen = ImageDataGenerator(rescale=1./255).flow_from_directory(
    os.path.join(split_dir, "test"),
    target_size=img_size,
    batch_size=batch_size,
    class_mode="categorical",
    shuffle=False
)

Copied 1801 images into split folders at: SPLIT_IMAGES
Found 1537 images belonging to 2 classes.
Found 221 images belonging to 2 classes.
Found 43 images belonging to 2 classes.


CNN

In [8]:
from tensorflow.keras import layers, models
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D

cnn_model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(224, 224,3)),
    layers.MaxPooling2D(2,2),
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),
    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(len(train_gen.class_indices), activation='softmax')
])

cnn_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
cnn_model.summary()

cnn_history = cnn_model.fit(
    train_gen,
    validation_data=val_gen,
    epochs= 10
)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │    11,075,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 2)              │           258 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,169,218 (42.61 MB)

 Trainable params: 11,169,218 (42.61 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
49/49 ━━━━━━━━━━━━━━━━━━━━ 25s 451ms/step - accuracy: 0.6468 - loss: 0.7019 - val_accuracy: 0.8416 - val_loss: 0.4783
Epoch 2/10
49/49 ━━━━━━━━━━━━━━━━━━━━ 20s 409ms/step - accuracy: 0.7526 - loss: 0.5003 - val_accuracy: 0.8416 - val_loss: 0.3802
Epoch 3/10
49/49 ━━━━━━━━━━━━━━━━━━━━ 19s 388ms/step - accuracy: 0.7816 - loss: 0.4794 - val_accuracy: 0.8552 - val_loss: 0.3575
Epoch 4/10
49/49 ━━━━━━━━━━━━━━━━━━━━ 20s 404ms/step - accuracy: 0.7918 - loss: 0.4636 - val_accuracy: 0.8597 - val_loss: 0.3661
Epoch 5/10
49/49 ━━━━━━━━━━━━━━━━━━━━ 19s 392ms/step - accuracy: 0.8179 - loss: 0.4272 - val_accuracy: 0.8597 - val_loss: 0.3049
Epoch 6/10
49/49 ━━━━━━━━━━━━━━━━━━━━ 20s 391ms/step - accuracy: 0.7858 - loss: 0.4638 - val_accuracy: 0.8009 - val_loss: 0.4031
Epoch 7/10
49/49 ━━━━━━━━━━━━━━━━━━━━ 20s 405ms/step - accuracy: 0.7928 - loss: 0.4259 - val_accuracy: 0.8597 - val_loss: 0.3991
Epoch 8/10
49/49 ━━━━━━━━━━━━━━━━━━━━ 19s 387ms/step - accuracy: 0.8004 - loss: 0.4177 - val_accu

VGG16

In [9]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras import layers, models
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D


base_model = VGG16(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False
vgg16_model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(len(train_gen.class_indices), activation='softmax')
])

vgg16_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
vgg16_model.summary()

vgg16_history = vgg16_model.fit(
    train_gen,
    validation_data=val_gen,
    epochs= 10
)


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ vgg16 (Functional)              │ (None, 7, 7, 512)      │    14,714,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 512)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 128)            │        65,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 2)              │           258 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,780,610 (56.38 MB)

 Trainable params: 65,922 (257.51 KB)

 Non-trainable params: 14,714,688 (56.13 MB)

Epoch 1/10
49/49 ━━━━━━━━━━━━━━━━━━━━ 27s 498ms/step - accuracy: 0.5992 - loss: 0.6816 - val_accuracy: 0.7104 - val_loss: 0.5380
Epoch 2/10
49/49 ━━━━━━━━━━━━━━━━━━━━ 22s 439ms/step - accuracy: 0.7148 - loss: 0.5516 - val_accuracy: 0.7376 - val_loss: 0.5122
Epoch 3/10
49/49 ━━━━━━━━━━━━━━━━━━━━ 22s 435ms/step - accuracy: 0.7132 - loss: 0.5312 - val_accuracy: 0.7195 - val_loss: 0.4945
Epoch 4/10
49/49 ━━━━━━━━━━━━━━━━━━━━ 22s 454ms/step - accuracy: 0.7285 - loss: 0.5207 - val_accuracy: 0.7149 - val_loss: 0.4762
Epoch 5/10
49/49 ━━━━━━━━━━━━━━━━━━━━ 23s 463ms/step - accuracy: 0.7180 - loss: 0.5275 - val_accuracy: 0.7195 - val_loss: 0.4748
Epoch 6/10
49/49 ━━━━━━━━━━━━━━━━━━━━ 23s 473ms/step - accuracy: 0.7583 - loss: 0.4847 - val_accuracy: 0.7421 - val_loss: 0.4717
Epoch 7/10
49/49 ━━━━━━━━━━━━━━━━━━━━ 23s 469ms/step - accuracy: 0.7398 - loss: 0.4967 - val_accuracy: 0.7511 - val_loss: 0.4644
Epoch 8/10
49/49 ━━━━━━━━━━━━━━━━━━━━ 22s 443ms/step - accuracy: 0.7411 - loss: 0.4878 - val_accu